In [1]:
%load_ext sql

In [2]:
import csv,sqlite3
import prettytable
prettytable.DEFAULT='DEFAULT'
conn=sqlite3.connect('my_data1.db')
cur=conn.cursor()

In [3]:
%sql sqlite:///my_data1.db

In [4]:
import pandas as pd
df = pd.read_csv("Spacex.csv")
df.to_sql("SPACEXTBL", conn, if_exists='replace', index=False,method="multi")
df.columns

Index(['Date', 'Time (UTC)', 'Booster_Version', 'Launch_Site', 'Payload',
       'PAYLOAD_MASS__KG_', 'Orbit', 'Customer', 'Mission_Outcome',
       'Landing_Outcome'],
      dtype='object')

In [5]:
#to remove blank rows from table
%sql DROP TABLE IF EXISTS SPACEXTABLE;

 * sqlite:///my_data1.db
Done.


[]

In [6]:
%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null

 * sqlite:///my_data1.db
Done.


[]

In [7]:
%%sql
--t1
select distinct "Launch_Site" from SPACEXTABLE;

 * sqlite:///my_data1.db
Done.


Launch_Site
CCAFS LC-40
VAFB SLC-4E
KSC LC-39A
CCAFS SLC-40


In [8]:
%%sql
--t2
select * from SPACEXTABLE
where Launch_Site like 'CCA%'
limit 5;

 * sqlite:///my_data1.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [13]:
%%sql
--t3
select sum("PAYLOAD_MASS__KG_") as total_payload_mass_kg from SPACEXTABLE where Customer='NASA (CRS)';

 * sqlite:///my_data1.db
Done.


total_payload_mass_kg
45596


In [18]:
%%sql
--t4
select avg("PAYLOAD_MASS__KG_") as avg_payload_mass_kg
from SPACEXTABLE where Booster_Version='F9 v1.1';

 * sqlite:///my_data1.db
Done.


avg_payload_mass_kg
2928.4


In [19]:
%%sql
--t5
select min(Date) as First_Successful_Ground_Pad_Landing
from SPACEXTABLE
where Landing_Outcome = 'Success (ground pad)';

 * sqlite:///my_data1.db
Done.


First_Successful_Ground_Pad_Landing
2015-12-22


In [20]:
%%sql
--t6
select Booster_Version from SPACEXTABLE where Landing_Outcome = 'Success (drone ship)' and "PAYLOAD_MASS__KG_" > 4000 and "PAYLOAD_MASS__KG_" < 6000;

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 FT B1022
F9 FT B1026
F9 FT B1021.2
F9 FT B1031.2


In [23]:
%%sql
--t7
select Mission_Outcome, count(Mission_Outcome) as Total_Count from SPACEXTABLE 
group by Mission_Outcome;

 * sqlite:///my_data1.db
Done.


Mission_Outcome,Total_Count
Failure (in flight),1
Success,98
Success,1
Success (payload status unclear),1


In [24]:
%%sql
--t8
select Booster_Version from SPACEXTABLE 
where "PAYLOAD_MASS__KG_" = (select max("PAYLOAD_MASS__KG_") from SPACEXTABLE);


 * sqlite:///my_data1.db
Done.


Booster_Version
F9 B5 B1048.4
F9 B5 B1049.4
F9 B5 B1051.3
F9 B5 B1056.4
F9 B5 B1048.5
F9 B5 B1051.4
F9 B5 B1049.5
F9 B5 B1060.2
F9 B5 B1058.3
F9 B5 B1051.6


In [26]:
%%sql
--t9
select substr(Date, 6, 2) as Month, "Landing_Outcome", Booster_Version, Launch_Site from SPACEXTABLE 
where Landing_Outcome = 'Failure (drone ship)' and substr(Date, 0, 5) = '2015';

 * sqlite:///my_data1.db
Done.


Month,Landing_Outcome,Booster_Version,Launch_Site
01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


In [27]:
%%sql
--t10
select "Landing_Outcome", count("Landing_Outcome") as Outcome_Count from SPACEXTABLE 
where Date between '2010-06-04' and '2017-03-20'
GROUP BY Landing_Outcome
order by Outcome_Count DESC;

 * sqlite:///my_data1.db
Done.


Landing_Outcome,Outcome_Count
No attempt,10
Success (drone ship),5
Failure (drone ship),5
Success (ground pad),3
Controlled (ocean),3
Uncontrolled (ocean),2
Failure (parachute),2
Precluded (drone ship),1
